Adding gender to authors metadata filter by ORCID

In [68]:
import pandas as pd

df = pd.read_csv("authors_metadata.csv")

df = df[~df["orcid"].isna()]
len(df.drop_duplicates("id"))

1014389

Removes honorific particules and checks whether it's a surname/name pair

In [69]:
import re

titled = df[df["display_name"].str.split(" ").str[0].str.contains(r"\b(?:Mr|Ms|Mrs|Dr|Prof)\.", regex=True)].copy()
titled["display_name"] = titled["display_name"].str.replace(r"\b(?:Mr|Ms|Mrs|Dr|Prof)\.", "", regex=True)

titled["display_name"] = titled["display_name"].str.replace(r"\.", "", regex=True)

words = titled["display_name"].str.strip().apply(
    lambda s: [w for w in re.split(r"\s+", s) if w.strip() != ""]
)

first_word = words.str[0].fillna("")
last_word  = words.str[-1].fillna("")  
num_words  = words.str.len()

keep = first_word.str.len() > 1

keep &= ~((num_words > 1) & (last_word.str.len() <= 1))
keep &= num_words > 1

titled = titled[keep]

Filter out initials only names, impossible to predict

In [70]:
df = df[~df["display_name"].str.split(" ").str[0].str.contains(r"\.")]
len(df.drop_duplicates("id"))

943042

Add back identifiable people with titles

In [72]:
df = pd.concat([df,titled])
len(df.drop_duplicates("id"))

943058

Pairs of name/surname from df, filter out doubles and lower, decode non ascii characters and non letter characters (excluding Mr/Ms since automatic)

In [73]:
import re
from unidecode import unidecode

names = pd.DataFrame(data={
    "name": df[~df["display_name"].str.split(" ").str[0].str.contains(r"\.")]["display_name"].str.split(" ").apply(lambda x: x[0] if isinstance(x, list) and len(x) >= 2 else None),
    "surname": df[~df["display_name"].str.split(" ").str[0].str.contains(r"\.")]["display_name"].str.split(" ").apply(lambda x: x[-1] if isinstance(x, list) and len(x) >= 2 else None)
})
names = names.drop_duplicates()
names["name"] = names["name"].str.lower()
names["surname"] = names["surname"].str.lower()
names["name"] = names["name"].fillna("").astype(str).apply(lambda x: unidecode(x))
names["name"] = names["name"].apply(lambda x: re.sub(r'[^a-z]', '', x) if isinstance(x, str) else x)
names["surname"] = names["surname"].fillna("").astype(str).apply(lambda x: unidecode(x))
names["surname"] = names["surname"].apply(lambda x: re.sub(r'[^a-z]', '', x) if isinstance(x, str) else x)
names

,name,surname
80,yixuan,lin
131,william,djakam
161,xin,du
207,zhiguo,zheng
410,zhifeng,tian
...,...,...
1634891,puja,gulati
1718382,,kadam
1723491,gvijaya,kumar
1927878,,jadhav


Only to clean the csv manually scraped from Forebears most popular surnames

In [ ]:
import os
import pandas as pd

# removing noise in the files
"""for file in os.listdir("surnames_by_countries") : 
    temp = pd.read_csv("surnames_by_countries/"+file, encoding_errors="replace")
    temp = temp[~temp.isna().any(axis=1)]
    temp.columns = temp.iloc[0].reset_index(drop=True)
    temp = temp[1:]
    temp["country"] = file.removesuffix(".csv")
    temp = temp.rename(columns={"Rank" : "rank","Surname":"surname","Incidence":"incidence","Frequency":"frequency"})
    temp.to_csv("./surnames_by_countries/"+file,index=False)
    """

Most popular surnames by country (Forebears)

In [74]:
import os
import pandas as pd

df_countries = pd.DataFrame()
# concatenating
for file in os.listdir("surnames_by_countries") : 
    temp = pd.read_csv("surnames_by_countries/"+file, encoding_errors="replace")
    df_countries = pd.concat([df_countries,temp], axis=0, ignore_index=True)
df_countries["surname"] = df_countries["surname"].str.lower()
df_countries["surname"] = df_countries["surname"].fillna("").astype(str).apply(lambda x: unidecode(x))
df_countries["surname"] = df_countries["surname"].apply(lambda x: re.sub(r'[^a-z\s]', '', x) if isinstance(x, str) else x)
df_countries.head()

,rank,surname,incidence,frequency,country
0,1,rekhviashvili,"2,203",1:110,abkhazia
1,2,maisuradze,"2,178",1:112,abkhazia
2,3,kvaratskhelia,"2,138",1:114,abkhazia
3,4,shonia,"1,738",1:140,abkhazia
4,5,gogokhia,"1,723",1:141,abkhazia


Wolrd Gender Name Dictionary, same preprocessing as FB and names

In [75]:
import re
from unidecode import unidecode
import pandas as pd

wgnd = pd.read_csv("WGND.csv")
wgnd["name"] = wgnd["name"].str.lower()
wgnd["name"]=wgnd["name"].fillna("").astype(str).apply(lambda x: unidecode(x))
wgnd["name"]=wgnd["name"].apply(lambda x: re.sub(r'[^a-z\s]', '', x) if isinstance(x, str) else x)

Adding language per country to df_countries to match with WGND langcode (more flexible than using the exact country match)

In [76]:
lang = pd.read_csv("surname_language.csv")
lang["country"] = lang["country"].str.removesuffix(".csv")
df_countries = df_countries.merge(lang, on="country", how="left")

For each pair, all possible languages and counts → 5.8M combinations

In [77]:
names = names.merge(df_countries[["surname","language", "incidence"]], how="left",on="surname")
names[~names["language"].isna()]

,name,surname,language,incidence
0,yixuan,lin,en,5
1,yixuan,lin,en,"9,114"
2,yixuan,lin,en,310
3,yixuan,lin,en,12
4,yixuan,lin,en,2
...,...,...,...,...
6140215,pavan,ka,ne,"35,013"
6140216,pavan,ka,tr,64
6140217,pavan,ka,fr,"211,276"
6140218,pavan,ka,zh,405


Fixing incidence (string convert to float when ',' meant thounsand)

In [78]:
def fix_thousands(val):
    val = str(val)
    if "," in val:
        # remove all commas except the last one
        parts = val.split(",")
        integer = "".join(parts[:-1])  # everything before last comma
        decimal = parts[-1].ljust(3, "0")  # last part, padded
        return float(integer + decimal)
    return float(val)
names["incidence"] = names["incidence"].apply(fix_thousands)
names[~names["language"].isna()]

,name,surname,language,incidence
0,yixuan,lin,en,5.0
1,yixuan,lin,en,9114.0
2,yixuan,lin,en,310.0
3,yixuan,lin,en,12.0
4,yixuan,lin,en,2.0
...,...,...,...,...
6140215,pavan,ka,ne,35013.0
6140216,pavan,ka,tr,64.0
6140217,pavan,ka,fr,211276.0
6140218,pavan,ka,zh,405.0


Split between consensual (same gender whatever the language) and non consensual gender in WGND

In [79]:
wgnd = wgnd.drop_duplicates().dropna()
gender_counts = wgnd.groupby("name")["gender"].nunique()

# Names with only one gender = consensual
consensual_names = gender_counts[gender_counts == 1].index
non_consensual_names = gender_counts[gender_counts > 1].index

df_consensual = wgnd[wgnd["name"].isin(consensual_names)].drop_duplicates("name")
df_non_consensual = wgnd[wgnd["name"].isin(non_consensual_names)]

print(len(df_consensual.drop_duplicates("name")))
print(len(df_non_consensual.drop_duplicates("name")))

2824110
15363


Merge consensual

In [80]:
consensual_match = names[["name","surname"]].merge(df_consensual[["name","gender"]],how="left",on="name").drop_duplicates().dropna()
consensual_match

,name,surname,gender
27,william,djakam,M
28,xin,du,F
34,zhiguo,zheng,M
44,zhifeng,tian,M
117,ricardo,ribeiro,M
...,...,...,...
6140158,erni,erawan,M
6140173,kram,charan,M
6140174,puja,gulati,F
6140177,gvijaya,kumar,F


Merge non consensual, takes language with most occurence of the name, if still two gender then no prediction

In [81]:
non_consensual_match = names.merge(df_non_consensual,how="left",left_on=["name","language"],right_on=["name","langcode"]).dropna()
non_consensual_match = non_consensual_match.sort_values("incidence", ascending=False).drop_duplicates(subset=["name", "surname", "language"], keep="first")

gender_counts = non_consensual_match.groupby("name")["gender"].nunique()
non_consensual = gender_counts[gender_counts > 1].index
non_consensual_match.loc[non_consensual_match["name"].isin(non_consensual), "gender"] = pd.NA
non_consensual_match = non_consensual_match.drop_duplicates(subset=["name", "surname", "language"]).dropna()
non_consensual_match

,name,surname,language,incidence,langcode,gender
1022623,yaling,wang,zh,105459482.0,zh,F
854336,changyu,wang,zh,105459482.0,zh,M
639391,yaling,li,zh,102565454.0,zh,F
317787,changyu,li,zh,102565454.0,zh,M
5832998,shoushan,li,zh,102565454.0,zh,M
...,...,...,...,...,...,...
5404658,raad,shareef,en,1.0,en,M
1437834,pilar,castillo,it,1.0,it,F
3866068,samy,sa,it,1.0,it,M
3866064,samy,sa,en,1.0,en,M


Merging names list and matched names

In [82]:
names = names.merge(pd.concat([consensual_match[["name","surname","gender"]],non_consensual_match[["name","surname","gender"]]], ignore_index=True),how="left",on=["name","surname"])
names = names.drop_duplicates(["name","surname"]).dropna(subset=["gender"])
names

,name,surname,language,incidence,gender
27,william,djakam,NaN,NaN,M
28,xin,du,km,1987.0,F
34,zhiguo,zheng,en,75.0,M
44,zhifeng,tian,zh,7201570.0,M
117,ricardo,ribeiro,ca,267.0,M
...,...,...,...,...,...
6323643,erni,erawan,NaN,NaN,M
6323658,kram,charan,hi,119263.0,M
6323659,puja,gulati,NaN,NaN,F
6323662,gvijaya,kumar,en,10255.0,F


Save file

In [83]:
names.to_csv("name_to_gender.csv", index=False)

Authors metadata preprocessing to count missing values

In [84]:
import re
from unidecode import unidecode

df = pd.read_csv("authors_metadata.csv")

df = df[~df["orcid"].isna()]

df["name"] = df["display_name"].str.split(" ").str[0].str.lower()
df["name"] = df["name"].fillna("").astype(str).apply(lambda x: unidecode(x))
df["name"] = df["name"].apply(lambda x: re.sub(r'[^a-z]', '', x) if isinstance(x, str) else x)
df["surname"] = df["display_name"].str.split(" ").str[-1].str.lower()
df["surname"] = df["surname"].fillna("").astype(str).apply(lambda x: unidecode(x))
df["surname"] = df["surname"].apply(lambda x: re.sub(r'[^a-z]', '', x) if isinstance(x, str) else x)
df

,Unnamed: 0,id,display_name,orcid,works_count,h_index,cited_by_count,topics,affiliations,name,surname
80,4586,A5051285025,Yixuan Lin,https://orcid.org/0000-0002-6888-520X,3,0,0,T11309_2;T12535_1;T11439_1;T13083_1;T10036_1;T...,I3132425485_2023_2022,yixuan,lin
131,8901,A5051407527,William Djakam,https://orcid.org/0000-0002-7652-3276,2,0,0,T10682_2;T10020_2;T10622_2,I32389192_2023,william,djakam
161,11270,A5051474754,Xin Du,https://orcid.org/0000-0001-5374-1433,1,0,0,T11273_1;T12592_1;T10064_1,NaN,xin,du
180,12516,A5051510277,G. V. Sri Rajiv Jegan,https://orcid.org/0009-0008-5318-5156,3,0,0,T10664_1;T11121_1;T11522_1;T10648_1;T10888_1;T...,I4210109528_2024_2023,g,jegan
207,14766,A5051574493,Zhiguo Zheng,https://orcid.org/0009-0004-3588-7729,6,0,0,T13996_2;T10941_2;T10667_1;T13702_1;T14287_1;T...,I20942203_2023,zhiguo,zheng
...,...,...,...,...,...,...,...,...,...,...,...
2614087,95267620,A5075109035,Chen Chen,https://orcid.org/0000-0002-5799-1330,1,0,0,T13734_1,I4210111970_2022,chen,chen
2614494,95464810,A5084455961,Maria Gul,https://orcid.org/0000-0001-6644-8455,3,0,0,T10215_1;T12768_1,NaN,maria,gul
2614771,95595343,A5090644335,iti kaushik,https://orcid.org/0000-0002-5061-7549,1,0,0,T10215_1,NaN,iti,kaushik
2614822,95617851,A5091705506,Miloš Švaňa,https://orcid.org/0000-0002-4227-4224,1,0,0,T12353_1;T10181_1,NaN,milos,svana


Merging authors metadata and gender prediction on name/surname pairs and adding Mr/Ms predictions

In [85]:
df = df.merge(names[["name","surname","gender"]],on=["name","surname"],how="left")
df.loc[df["display_name"].str.contains(r"\bMr\.", regex=True, na=False), "gender"] = "M"
df.loc[df["display_name"].str.contains(r"\bMs\.", regex=True, na=False), "gender"] = "F"
df[~df["gender"].isna()]

,Unnamed: 0,id,display_name,orcid,works_count,h_index,cited_by_count,topics,affiliations,name,surname,gender
1,8901,A5051407527,William Djakam,https://orcid.org/0000-0002-7652-3276,2,0,0,T10682_2;T10020_2;T10622_2,I32389192_2023,william,djakam,M
2,11270,A5051474754,Xin Du,https://orcid.org/0000-0001-5374-1433,1,0,0,T11273_1;T12592_1;T10064_1,NaN,xin,du,F
4,14766,A5051574493,Zhiguo Zheng,https://orcid.org/0009-0004-3588-7729,6,0,0,T13996_2;T10941_2;T10667_1;T13702_1;T14287_1;T...,I20942203_2023,zhiguo,zheng,M
5,29320,A5051990109,Zhifeng Tian,https://orcid.org/0009-0006-4358-7667,3,0,0,T12601_1;T10846_1;T10388_1;T10270_1;T10767_1;T...,I2801045807_2023;I74872605_2020_2009,zhifeng,tian,M
7,39567,A5052282032,Ricardo Ribeiro,https://orcid.org/0000-0003-2983-9080,2,0,0,T12709_1;T12205_1;T13018_1;T13650_1;T10551_1;T...,I4210091481_2024_2023,ricardo,ribeiro,M
...,...,...,...,...,...,...,...,...,...,...,...,...
1014382,95005523,A5062684628,Linda Wiechetek,https://orcid.org/0000-0002-5171-0841,1,0,0,T12881_1;T10181_1,NaN,linda,wiechetek,F
1014383,95109692,A5067617469,Dmitriy S. Tyagunov,https://orcid.org/0000-0002-8729-7518,1,0,0,T10320_1,I4210126723_2023,dmitriy,tyagunov,M
1014385,95464810,A5084455961,Maria Gul,https://orcid.org/0000-0001-6644-8455,3,0,0,T10215_1;T12768_1,NaN,maria,gul,F
1014387,95617851,A5091705506,Miloš Švaňa,https://orcid.org/0000-0002-4227-4224,1,0,0,T12353_1;T10181_1,NaN,milos,svana,M


In [87]:
df[~df["gender"].isna()]["id"].nunique()/df["id"].nunique()

0.707742296101397

In [88]:
df[~df["gender"].isna()]["display_name"].nunique()/df["display_name"].nunique()

0.7144835678830683

In [ ]:
df[~df["gender"].isna()]["display_name"].nunique()/df["display_name"].nunique()

In [90]:
len(df[~df["gender"].isna()].drop_duplicates(subset=["name","surname"]))/756736

0.759448473443843

In [91]:
df.drop(columns=df.columns[0]).to_csv("authors_metadata_enhanced.csv",index=False)

AI test

In [1]:
import pandas as pd

df = pd.read_csv("authors_metadata_enhanced.csv")

In [16]:
df[~df["gender"].isna()][["display_name","name","gender"]].drop_duplicates(subset="name").sample(10000).to_csv("gender_sample.csv",index=False)


Test ? seems like not enough pairs

In [94]:
import pandas as pd

gio = pd.DataFrame()

for file in ["gender_proba.csv","gender_proba_2.csv","gender_proba_3.csv"] : 
    temp = pd.read_csv(file, encoding="latin-1")
    gio = pd.concat([gio,temp])

In [ ]:
gio["gender"] = gio["gender"].map({"male" : "M", "female" : "F"})
gio = gio.drop_duplicates(subset="given_name")

In [100]:
gio

,id,gender,given_name,proba,country_id
0,1473.0,M,fei,"0,57",all
1,50118.0,M,wang,"0,97",all
2,6952.0,F,li,"0,66",all
3,4340.0,M,yang,"0,66",all
4,7368.0,F,gao,"0,95",all
...,...,...,...,...,...
13088,257.0,M,pb,"0,77",all
13089,55.0,M,iy,"0,73",all
13090,9501.0,F,ayten,"0,98",all
13091,766.0,M,vas,"0,77",all
